In [ ]:
#31 july: Parses InPTA DR2 .par files (github.com/inpta/InPTA.DR2.NA) into per-pulsar geometry + white-noise table. Validated: 11 pulsars with baseline <2.5 yr, consistent with Nobleson et al. 2025 (arXiv:2512.20455).

In [6]:
import sys
!{sys.executable} -m pip install pandas astropy

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 882.6 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.3 MB 838.9 kB/s eta 0:00:13
   -- ------------------------------------- 0.8/11.3 MB 838.9 kB/s eta 0:00:13
   --- ------------------------------------ 1.0/11.3 MB 867.1 kB/s eta 0:00:12
   ---- ----------------------------------- 1.3/11.3 MB 932.1 kB/s eta 0:00:11
   ----- ---------------------------------- 1.6/11.3 MB 975.2 kB/s eta 0:00:11
   ------ --------------------------------- 1.8/11.3 MB 967.9 kB/s eta 0:00:10
   ------- ------------------

In [7]:
# 01_parse_inpta_dr2_positions_timing.ipynb
import subprocess, glob, os
import numpy as np, pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# --- clone the InPTA DR2 noise-analysis par-file repo (once) ---
if not os.path.isdir("InPTA.DR2.NA"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/inpta/InPTA.DR2.NA.git"], check=True)

def parse_par(f):
    d={}
    for line in open(f):
        p=line.split()
        if p: d[p[0]] = p[1:] if len(p)>1 else ['']
    name = d.get('PSRJ',[os.path.basename(f)])[0]
    if 'RAJ' in d:
        c = SkyCoord(d['RAJ'][0], d['DECJ'][0], unit=(u.hourangle,u.deg), frame='icrs')
    else:  # 12 pulsars use ecliptic coords
        c = SkyCoord(lon=float(d['ELONG'][0])*u.deg, lat=float(d['ELAT'][0])*u.deg,
                     frame='barycentricmeanecliptic').icrs
    start,fin = float(d['START'][0]), float(d['FINISH'][0])
    span = (fin-start)/365.25
    ntoa = int(d['NTOA'][0]) if 'NTOA' in d else np.nan
    tres = float(d['TRES'][0]) if 'TRES' in d else np.nan   # microseconds
    return dict(psr=name, ra_deg=c.ra.deg, dec_deg=c.dec.deg,
                phi=c.ra.rad, theta=np.pi/2-c.dec.rad,     # hasasia convention
                span_yr=span, mjd_start=start, mjd_finish=fin,
                ntoa=ntoa, tres_us=tres, toa_per_yr=ntoa/span)

rows = [parse_par(f) for f in sorted(glob.glob("InPTA.DR2.NA/par/*.par"))]
df = pd.DataFrame(rows).sort_values("span_yr", ascending=False).reset_index(drop=True)

# validation checkpoint: should be 11, matching DR2 II abstract
n_short = (df.span_yr < 2.5).sum()
print(f"{len(df)} pulsars parsed; {n_short} with baseline <2.5 yr (paper says 11)")
assert n_short == 11, "mismatch — investigate before proceeding"

df.to_csv("inpta_dr2_geometry_whitenoise.csv", index=False)
df
print(df.to_string())
print(f"\nShape: {df.shape}")
print(f"CSV saved to: {os.path.abspath('inpta_dr2_geometry_whitenoise.csv')}")

27 pulsars parsed; 11 with baseline <2.5 yr (paper says 11)
           psr      ra_deg    dec_deg       phi     theta   span_yr     mjd_start    mjd_finish   ntoa  tres_us   toa_per_yr
0   J1857+0943  284.401620   9.721429  4.963745  1.401125  7.199873  57768.330367  60398.083969   1868    8.622   259.449022
1   J1730-2304  262.590350 -23.075324  4.583066  1.973537  7.199664  57768.291130  60397.968576   5585   12.884   775.730595
2   J1909-3744  287.447603 -37.737474  5.016907  2.229440  7.199657  57768.350390  60398.025058   4160    3.470   577.805315
3   J0751+1807  117.788138  18.127311  2.055791  1.254415  7.188856  57773.863121  60399.592593   1518   18.156   211.160177
4   J1022+1001  155.741751  10.031690  2.718206  1.395710  7.188649  57773.906756  60399.560937   3346   12.162   465.456003
5   J0613-0200   93.433238  -2.013149  1.630718  1.605932  7.185894  57773.843792  60398.491493   5028    2.705   699.704193
6   J1012+5307  153.139337  53.117228  2.672786  0.643726  7.1014